## Parallel Workflows

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [6]:
class BatsmanState(TypedDict):
    runs: int
    balls :int
    fours :int
    sixes :int

    sr: float
    bpb : float
    boundary_percent :float
    summary :str

In [14]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    runs = state['runs']
    balls = state['balls']

    sr = (runs/balls)*100
    return {'sr':sr}

In [16]:
def calculate_bpb(state:BatsmanState) -> BatsmanState:
    balls = state['balls']
    fours = state['fours']
    sixes = state['sixes']

    bpb = balls/(fours+sixes)
    return {'bpb':bpb}


In [20]:
def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    fours = state['fours']
    sixes = state['sixes']
    runs  = state['runs']
    runs_boundary = (fours*4 + sixes*6)

    boundary_percent = (runs_boundary/runs)*100

    return {'boundary_percent':boundary_percent}


In [17]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
Strike Rate = {state['sr']} \n
Balls per Boundary = {state['bpb']} \n
Boundary Percent = {state['boundary_percent']}
"""
    
    return {'summary':summary}

In [21]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()


In [22]:
initial_state = {
    'runs' :100,
    "balls" : 50,
    'fours' : 6,
    'sixes' : 4
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nStrike Rate = 200.0 \n\nBalls per Boundary = 5.0 \n\nBoundary Percent = 48.0\n'}